# Eksperimen 22: Inovasi Agregator GNN: Elementwise Maximization vs Mean (Google Colab)

Eksekusi pipeline eksperimen EXP22 pada Google Colab dengan GPU T4/A100:
- Komparasi langsung SAGEConv `aggr='mean'` vs `aggr='max'`.
- Visualisasi perbandingan embedding t-SNE.


In [ ]:
%%writefile config.colab.yaml
data:
  transactions: "/content/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01" # Tanggal T0 (era modern 2016-2020 dengan sebaran fraud stabil)
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

sampling:
  method: "uniform" # Baseline sampling kontrol (Uniform)
  undersampling_ratio: 1.0 # 1.0 = tidak ada undersampling (mempertahankan topologi asli)

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "max" # Pilihan: max, mean, sum

training:
  batch_size: 1024
  epochs: 10
  learning_rate: 0.001
  num_neighbors: [15, 10]

classifier:
  n_estimators: 100
  max_depth: 5
  learning_rate: 0.05

output:
  model_dir: "/content/model/exp22"
  result_dir: "/content/result/exp22"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = 'config.reference.yaml') -> dict:
    """
    Memuat file konfigurasi YAML untuk EXP22.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Konfigurasi tidak ditemukan di: {path.resolve()}")
        
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path

def load_and_preprocess_data(file_path: str, start_year: int = 2016, nrows: int = None) -> tuple:
    """
    Memuat dataset IBM Credit Card Transactions dan memfilter era modern (Year >= start_year).
    Melakukan parsing timestamp kronologis, pembersihan nominal Amount, encoding target,
    dan transformasi kategorikal.
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
    
    # Jika start_year ditentukan dan nrows None/besar, gunakan chunking efisien untuk melewati tahun 1991-2015
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            # Filter baris dengan tahun >= start_year
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    
    print("Encoding target label...")
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    print("Encoding categorical features...")
    for col in cat_columns:
        df[col] = df[col].fillna('<missing>').astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        
    # Scale Amount menggunakan StandardScaler
    scaler = StandardScaler()
    df['Amount'] = scaler.fit_transform(df[['Amount']])

    # Fitur numerik/kategori untuk node Transaction
    feature_cols = ['Amount'] + cat_columns
    
    return df, feature_cols

def get_temporal_window_split(
    df: pd.DataFrame,
    reference_date: str = '2018-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60
) -> tuple:
    """
    Membagi dataset berdasarkan protokol rolling window SHINE FRAUD II:
    - Train Window      : [T0, T0 + train_days)
    - Validation Window : [T0 + train_days, T0 + train_days + val_days)
    - Embargo Gap       : [T0 + train_days + val_days, T0 + train_days + val_days + gap_days)
    - Test Window       : [T0 + train_days + val_days + gap_days, T0 + train_days + val_days + gap_days + test_days)
    
    Mengembalikan: (train_df, val_df, test_df, gap_df, stats_dict)
    """
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def generate_rolling_window_indices(
    df: pd.DataFrame,
    start_date: str = '2016-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
    step_days: int = 30
):
    """
    Generator untuk memproduksi beberapa timeframe jendela bergulir (rolling windows).
    """
    total_window_span = timedelta(days=train_days + val_days + gap_days + test_days)
    curr_t0 = pd.to_datetime(start_date)
    max_datetime = df['datetime'].max()
    
    fold_idx = 0
    while curr_t0 + total_window_span <= max_datetime:
        fold_idx += 1
        yield fold_idx, curr_t0.strftime('%Y-%m-%d')
        curr_t0 += timedelta(days=step_days)



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp22 & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list) -> HeteroData:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi tabular (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi (scaled)
      - merchant: Agregat rata-rata Amount & jumlah transaksi (scaled)
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan user_mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id')
    
    scaler_user = StandardScaler()
    user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    
    # Pastikan urutan sesuai dengan merchant_mapping
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id')
    
    scaler_merchant = StandardScaler()
    merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    data['user'].x = user_features
    data['merchant'].x = merchant_features
    
    # 3. Fitur dan label Transaction
    x_tx = torch.tensor(df[feature_cols].values, dtype=torch.float)
    y_tx = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].x = x_tx
    data['transaction'].y = y_tx
    data['transaction'].time = torch.arange(num_transactions)
    
    # 4. Konstruksi Edges (Bipartit / Tripartit dua arah)
    user_src = [user_mapping[uid] for uid in df['User']]
    tx_dst = list(range(num_transactions))
    edge_index_user_tx = torch.tensor([user_src, tx_dst], dtype=torch.long)
    
    merchant_src = [merchant_mapping[mid] for mid in df['Merchant Name']]
    edge_index_merchant_tx = torch.tensor([merchant_src, tx_dst], dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_index_user_tx
    data['merchant', 'sells', 'transaction'].edge_index = edge_index_merchant_tx
    
    # Reverse edges untuk propagasi pesan dua arah pada GraphSAGE
    data['transaction', 'rev_buys', 'user'].edge_index = edge_index_user_tx.flip([0])
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_index_merchant_tx.flip([0])
    
    print(f"Graf selesai dibangun: {num_users} users, {num_merchants} merchants, {num_transactions} transaksi.")
    return data



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Jaccard Similarity / Local Structure Proxy).
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Bobot struktural berbasis inverse degree
        weights = 1.0 / (deg_src[src_nodes] + 1e-5)
        weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-5)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Degree Centrality Proxy).
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        importance_weights = deg_src[src_nodes]
        importance_weights = (importance_weights - importance_weights.min()) / (importance_weights.max() - importance_weights.min() + 1e-5)
        data[edge_type].edge_weight = importance_weights + 0.1
    return data

def get_neighbor_loader(data: HeteroData, batch_size: int = 1024, num_neighbors: list = [15, 10], method: str = 'uniform', shuffle: bool = True, input_nodes=None, num_workers: int = 0):
    """
    Mengembalikan PyG NeighborLoader berdasarkan strategi neighbor sampling yang dipilih.
    """
    if method == 'uniform':
        data = apply_uniform_sampling(data)
    elif method == 'topology':
        data = apply_topology_aware_sampling(data)
    elif method == 'importance':
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {method}")

    if input_nodes is None:
        input_nodes = ('transaction', torch.arange(data['transaction'].num_nodes))

    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,
        shuffle=shuffle,
        weight_attr='edge_weight',
        num_workers=num_workers
    )
    return loader



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    """
    Arsitektur Heterogeneous GraphSAGE untuk Klasifikasi Transaksi Fraud.
    Mendukung fungsi agregasi fleksibel:
    - 'max'  : Elementwise Maximization (SHINE Eq. 10 & 11) untuk mencegah minority feature dilution.
    - 'mean' : Standar GraphSAGE averaging (Baseline).
    - 'sum'  : Elementwise summation.
    """
    def __init__(self, hidden_channels: int, out_channels: int, num_layers: int, data_metadata: tuple, feature_dims: dict, aggr: str = 'max'):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal ke hidden_channels
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
                
        # Node tanpa fitur jika ada
        for node_type, param in self.param_embeddings.items():
            num_nodes = num_nodes_dict[node_type]
            h_dict[node_type] = param.expand(num_nodes, -1)
                
        # 2. Message Passing dengan agregator terpilih (max / mean / sum)
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            # ReLU dan BatchNorm pada transaction nodes
            for node_type in h_dict.keys():
                h_dict[node_type] = F.relu(h_dict[node_type])
                if node_type == 'transaction':
                    h_dict[node_type] = self.bns[i](h_dict[node_type])
        
        if return_embeddings:
            return h_dict['transaction']
            
        # 3. Klasifikasi biner untuk transaction nodes
        out = self.classifier(h_dict['transaction'])
        return torch.sigmoid(out).squeeze(-1)

def manual_weighted_bce(probs, targets, pos_weight):
    """
    Class-weighted Binary Cross-Entropy Loss untuk output probabilitas sigmoid.
    pos_weight = (num_negatives / (num_positives + 1e-7))
    """
    loss = - (pos_weight * targets * torch.log(probs + 1e-7) + 
              (1.0 - targets) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()



In [ ]:
%%writefile visualize_tsne.py
import argparse
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg') # Headless rendering
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def load_sample_and_metrics(result_dir: Path, aggr: str):
    """
    Memuat sample embedding .npz dan file hasil .json untuk aggregator tertentu.
    """
    sample_file = result_dir / f"exp22_{aggr}_embeddings_sample.npz"
    result_file = result_dir / f"exp22_{aggr}_results.json"
    
    if not sample_file.exists():
        return None, None
        
    data = np.load(sample_file)
    embeddings = data['embeddings']
    labels = data['labels']
    
    metrics = None
    if result_file.exists():
        with open(result_file, 'r') as f:
            metrics = json.load(f)
            
    return (embeddings, labels), metrics

def calculate_cluster_separation(embeddings, labels):
    """
    Menghitung rasio separasi centroid antar-kelas vs variansi intra-kelas
    (Euclidean Distance between Fraud Centroid & Normal Centroid).
    """
    fraud_mask = (labels == 1)
    normal_mask = (labels == 0)
    
    if not np.any(fraud_mask) or not np.any(normal_mask):
        return 0.0
        
    c_fraud = embeddings[fraud_mask].mean(axis=0)
    c_normal = embeddings[normal_mask].mean(axis=0)
    
    dist = np.linalg.norm(c_fraud - c_normal)
    return float(dist)

def plot_tsne_comparison(result_dir: Path, output_file: Path):
    """
    Membuat visualisasi komparatif t-SNE 2D untuk agregator Mean vs Max (dan Sum jika ada).
    """
    aggregators = ['mean', 'max']
    # Cek apakah 'sum' juga ada
    if (result_dir / "exp22_sum_embeddings_sample.npz").exists():
        aggregators.append('sum')
        
    loaded_data = {}
    for aggr in aggregators:
        data_tuple, metrics = load_sample_and_metrics(result_dir, aggr)
        if data_tuple is not None:
            loaded_data[aggr] = (data_tuple, metrics)
            
    if not loaded_data:
        print(f"[WARNING] Tidak ditemukan berkas embedding sample (exp22_*_embeddings_sample.npz) di: {result_dir}")
        print("Pastikan pipeline training (main.py) telah dijalankan terlebih dahulu.")
        return False

    print(f"Menjalankan t-SNE embedding comparison untuk: {list(loaded_data.keys())}...")
    
    n_plots = len(loaded_data)
    fig, axes = plt.subplots(1, n_plots, figsize=(7.5 * n_plots, 6.5), dpi=300)
    if n_plots == 1:
        axes = [axes]
        
    fig.patch.set_facecolor('#FFFFFF')
    
    for idx, (aggr, (data_tuple, metrics)) in enumerate(loaded_data.items()):
        ax = axes[idx]
        embeddings, labels = data_tuple
        
        print(f"Menghitung t-SNE untuk {aggr.upper()} ({len(labels)} sampel)...")
        tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
        emb_2d = tsne.fit_transform(embeddings)
        
        sep_dist = calculate_cluster_separation(emb_2d, labels)
        
        normal_idx = np.flatnonzero(labels == 0)
        fraud_idx = np.flatnonzero(labels == 1)
        
        # Plot titik transaksi normal
        ax.scatter(
            emb_2d[normal_idx, 0], 
            emb_2d[normal_idx, 1], 
            c='#64748B', 
            alpha=0.25, 
            s=16, 
            label=f'Normal (N={len(normal_idx)})',
            edgecolors='none'
        )
        
        # Plot titik transaksi fraud (merah menyala dengan outline halus)
        ax.scatter(
            emb_2d[fraud_idx, 0], 
            emb_2d[fraud_idx, 1], 
            c='#E63946', 
            alpha=0.92, 
            s=48, 
            label=f'Fraud (N={len(fraud_idx)})',
            edgecolors='#1E293B',
            linewidths=0.6,
            zorder=5
        )
        
        # Format Subtitle dengan Metrik Evaluasi jika tersedia
        title_aggr = "Baseline: Mean Aggregator" if aggr == 'mean' else ("Proposed: Max Pooling (Elementwise Max)" if aggr == 'max' else "Ablation: Sum Aggregator")
        metric_str = ""
        if metrics:
            opt = metrics.get('metrics_optimal_threshold', {})
            ind = metrics.get('metrics_threshold_independent', {})
            f1 = opt.get('f1_score', 0.0)
            rec = opt.get('recall', 0.0) * 100
            aucpr = ind.get('auprc', 0.0)
            metric_str = f"\nAUCPR: {aucpr:.4f} | F1: {f1:.4f} | Recall: {rec:.1f}%"
            
        ax.set_title(f"{title_aggr}{metric_str}\nCentroid Separation: {sep_dist:.2f}", fontsize=12, fontweight='bold', pad=12)
        ax.set_xlabel("t-SNE Dimension 1", fontsize=10, fontweight='medium')
        ax.set_ylabel("t-SNE Dimension 2", fontsize=10, fontweight='medium')
        ax.grid(True, linestyle='--', alpha=0.35)
        ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
        ax.set_facecolor('#F8FAFC')
        
        # Highlight anotasi temuan
        if aggr == 'mean':
            ax.text(0.03, 0.05, "Minority Feature Dilution:\nFraud tenggelam di klaster normal", 
                    transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#FEF2F2", ec="#FCA5A5", alpha=0.9))
        elif aggr == 'max':
            ax.text(0.03, 0.05, "Anomaly Preservation:\nFraud terpisah jelas & terkonsentrasi", 
                    transform=ax.transAxes, fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#F0FDF4", ec="#86EFAC", alpha=0.9))

    plt.suptitle("Eksperimen 22: Separabilitas Representasi Embedding Transaksi (t-SNE)\n"
                 "Pengaruh Fungsi Agregasi GNN terhadap Ketahanan Minority Feature Dilution", 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"\n[SUCCESS] Visualisasi komparatif t-SNE tersimpan ke: {output_file}\n")
    return True

def main():
    parser = argparse.ArgumentParser(description="EXP22: Visualisasi komparasi t-SNE separabilitas embedding GNN")
    parser.add_argument('--result-dir', type=str, default='../../result/exp22', help='Direktori penyimpanan hasil exp22')
    parser.add_argument('--output-file', type=str, default='embeddings_tsne_comparison.png', help='Nama file gambar output')
    args = parser.parse_args()
    
    result_dir = Path(args.result_dir)
    output_file = result_dir / args.output_file
    plot_tsne_comparison(result_dir, output_file)

if __name__ == '__main__':
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_and_preprocess_data, get_temporal_window_split
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur orisinal transaksi.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set.
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 0.51, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(precision_score(val_labels, preds, zero_division=0)),
                "val_recall": float(recall_score(val_labels, preds, zero_division=0))
            }
            
    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def main():
    parser = argparse.ArgumentParser(description="Eksperimen 22: GNN Aggregator Innovation (Max Pooling vs Mean)")
    parser.add_argument('--config', type=str, default='config.reference.yaml', help='Path ke file konfigurasi YAML')
    parser.add_argument('--aggr', type=str, choices=['max', 'mean', 'sum'], default=None, help='Fungsi agregasi pesan HeteroConv (max, mean, sum)')
    parser.add_argument('--sampling', type=str, choices=['uniform', 'topology', 'importance'], default=None, help='Metode neighbor sampling')
    parser.add_argument('--reference-date', type=str, default=None, help='Override tanggal T0 window (format: YYYY-MM-DD)')
    parser.add_argument('--nrows', type=int, default=None, help='Limit baris data untuk debugging lokal')
    parser.add_argument('--start-year', type=int, default=None, help='Override filter tahun awal dataset')
    parser.add_argument('--sample-size', type=int, default=5000, help='Jumlah sampel normal untuk visualisasi t-SNE')
    args = parser.parse_args()
    
    total_pipeline_start = time.time()
    
    # 1. Load Konfigurasi
    cfg = load_config(args.config)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Ambil parameter aggr dari CLI jika diberikan, atau dari file config
    aggr = args.aggr if args.aggr is not None else cfg.get('model', {}).get('aggr', 'max')
    sampling_method = args.sampling if args.sampling is not None else cfg.get('sampling', {}).get('method', 'uniform')
    
    print(f"\n{'='*80}")
    print(f"  EXP22: GNN AGGREGATOR INNOVATION: ELEMENTWISE MAXIMIZATION VS MEAN (SHINE)")
    print(f"{'='*80}")
    print(f"Device               : {device}")
    print(f"Aggregator           : {aggr.upper()} (SAGEConv aggr='{aggr}')")
    print(f"Sampling Method      : {sampling_method}")
    
    # Override time_window parameters jika disediakan di CLI
    tw_cfg = cfg.get('time_window', {})
    reference_date = args.reference_date if args.reference_date is not None else tw_cfg.get('reference_date', '2018-01-01')
    train_days = tw_cfg.get('train_days', 300)
    val_days = tw_cfg.get('val_days', 30)
    gap_days = tw_cfg.get('gap_days', 8)
    test_days = tw_cfg.get('test_days', 60)
    
    start_year = args.start_year if args.start_year is not None else cfg['data'].get('start_year', 2016)
    dataset_path = cfg['data']['transactions']
    nrows = args.nrows if args.nrows is not None else cfg['data'].get('max_rows', None)
    
    # Siapkan direktori output
    model_dir = Path(cfg.get('output', {}).get('model_dir', '../../model/exp22'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', '../../result/exp22'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 2. Preprocessing Data & Filter Temporal 2016-2020
    print("\n--- [Langkah 1: Preprocessing Data (2016-2020)] ---")
    data_prep_start = time.time()
    df, feature_cols = load_and_preprocess_data(dataset_path, start_year=start_year, nrows=nrows)
    data_prep_time = time.time() - data_prep_start
    print(f"Data preprocessing selesai dalam {data_prep_time:.2f} detik. Total baris: {len(df)}")
    
    # 3. Pembagian Temporal Window (300d train, 30d val, 8d gap, 60d test)
    print("\n--- [Langkah 2: Pembagian Jendela Temporal SHINE FRAUD II] ---")
    train_df, val_df, test_df, gap_df, window_stats = get_temporal_window_split(
        df, 
        reference_date=reference_date,
        train_days=train_days,
        val_days=val_days,
        gap_days=gap_days,
        test_days=test_days
    )
    
    print("\nStatistik Partisi Jendela Temporal:")
    for split_key, sp in window_stats['splits'].items():
        print(f"  • {sp['name']:12s}: {sp['num_transactions']:8d} transaksi | "
              f"{sp['num_fraud']:5d} fraud ({sp['fraud_rate_pct']:6.3f}%) | "
              f"Rentang: {sp['start_date'][:10]} s/d {sp['end_date'][:10]}")
              
    # Simpan window_statistics.json
    stats_path = result_dir / "window_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(window_stats, f, indent=4)
    print(f"Statistik jendela waktu tersimpan ke: {stats_path}")
    
    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError("Dataframe Train atau Test kosong! Periksa kembali parameter reference_date atau nrows.")

    # 4. Undersampling (Default: 1.0 = Tanpa Undersampling)
    undersample_ratio = cfg['sampling'].get('undersampling_ratio', 1.0)
    train_df_sampled = graph_level_undersample(train_df, ratio=undersample_ratio)

    # 5. Konstruksi Graf Heterogen (PyG)
    print("\n--- [Langkah 3: Membangun Graf Train] ---")
    train_data = build_hetero_graph(train_df_sampled, feature_cols)
    
    batch_size = cfg['training'].get('batch_size', 1024)
    num_neighbors = cfg['training'].get('num_neighbors', [15, 10])
    
    train_loader = get_neighbor_loader(
        train_data, 
        batch_size=batch_size, 
        num_neighbors=num_neighbors,
        method=sampling_method, 
        shuffle=True
    )
    
    # 6. Definisi Arsitektur Model GNN (HeteroGraphSAGE dengan aggr terpilih)
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 2,
        'merchant': 2
    }
    hidden_channels = cfg['model'].get('hidden_channels', 128)
    out_channels = cfg['model'].get('out_channels', 1)
    num_layers = cfg['model'].get('num_layers', 2)
    
    model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_layers=num_layers,
        data_metadata=train_data.metadata(),
        feature_dims=feature_dims,
        aggr=aggr
    ).to(device)
    
    num_pos = int(train_data['transaction'].y.sum().item())
    num_neg = int(train_data['transaction'].num_nodes - num_pos)
    pos_weight = torch.tensor([num_neg / (num_pos + 1e-7)]).to(device)
    print(f"GNN Aggregator: '{aggr}' | Class-Weight pos_weight: {pos_weight.item():.2f}")
    
    lr = cfg['training'].get('learning_rate', 0.001)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 7. Phase 1: Pelatihan GraphSAGE Encoder
    epochs = cfg['training'].get('epochs', 10)
    print(f"\n--- [Phase 1: Training GraphSAGE Encoder ({aggr.upper()} Aggregator, {epochs} Epochs)] ---")
    gnn_train_start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {avg_loss:.4f}")
    gnn_train_time = time.time() - gnn_train_start
    print(f"Pelatihan GNN selesai dalam {gnn_train_time:.2f} detik.")
    
    # Simpan model GNN state dict sebelum membebaskan memori
    gnn_model_path = model_dir / f"exp22_{aggr}_gnn.pt"
    torch.save(model.state_dict(), gnn_model_path)
    print(f"Model GNN tersimpan di: {gnn_model_path}")

    # Bebaskan train_loader dan optimizer
    del train_loader, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    # 8. Phase 2: Ekstraksi Fitur (Train, Validation, Test)
    print("\n--- [Phase 2: Ekstraksi Fitur Representasi Induktif] ---")
    orig_dim = len(feature_cols)
    emb_dim = hidden_channels
    
    # 8a. Ekstraksi fitur Train
    print(f"Mengekstrak fitur Train ({len(train_df_sampled)} sampel)...")
    train_extract_loader = get_neighbor_loader(
        train_data, 
        batch_size=4096, 
        num_neighbors=num_neighbors, 
        method=sampling_method, 
        shuffle=False
    )
    train_features, train_labels, inf_time_train = extract_features(
        model, train_extract_loader, device,
        total_samples=len(train_df_sampled),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del train_data, train_extract_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    # 8b. Ekstraksi fitur Validation (hanya mencakup Train + Val tanpa Test agar bebas data leakage)
    print(f"Mengekstrak fitur Validation ({len(val_df)} sampel)...")
    val_graph_df = pd.concat([train_df, val_df], ignore_index=True)
    val_data = build_hetero_graph(val_graph_df, feature_cols)
    val_node_indices = torch.arange(len(train_df), len(val_graph_df))
    
    val_loader = get_neighbor_loader(
        val_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', val_node_indices)
    )
    val_features, val_labels, inf_time_val_gnn = extract_features(
        model, val_loader, device,
        total_samples=len(val_node_indices),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del val_data, val_loader, val_graph_df
    gc.collect()
    torch.cuda.empty_cache()

    # 8c. Ekstraksi fitur Test (mencakup Train + Val + Test)
    print(f"Mengekstrak fitur Test ({len(test_df)} sampel)...")
    test_graph_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    test_data = build_hetero_graph(test_graph_df, feature_cols)
    test_node_indices = torch.arange(len(train_df) + len(val_df), len(test_graph_df))
    
    test_loader = get_neighbor_loader(
        test_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', test_node_indices)
    )
    test_features, test_labels, inf_time_test_gnn = extract_features(
        model, test_loader, device,
        total_samples=len(test_node_indices),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del test_data, test_loader, test_graph_df, model
    gc.collect()
    torch.cuda.empty_cache()

    # 8d. Simpan sampel representasi embedding Test untuk visualisasi t-SNE separabilitas kelas
    print(f"\n--- [Menyiapkan Sampel Embedding Test untuk t-SNE (EXP22)] ---")
    test_emb_only = test_features[:, orig_dim:]
    fraud_indices = np.flatnonzero(test_labels == 1)
    normal_indices = np.flatnonzero(test_labels == 0)
    
    rng = np.random.default_rng(42)
    sample_normal_size = min(args.sample_size, len(normal_indices))
    selected_normal = rng.choice(normal_indices, size=sample_normal_size, replace=False)
    
    combined_sample_idx = np.concatenate([selected_normal, fraud_indices])
    rng.shuffle(combined_sample_idx)
    
    sample_embeddings = test_emb_only[combined_sample_idx]
    sample_labels = test_labels[combined_sample_idx]
    
    emb_sample_path = result_dir / f"exp22_{aggr}_embeddings_sample.npz"
    np.savez_compressed(
        emb_sample_path,
        embeddings=sample_embeddings,
        labels=sample_labels,
        num_fraud=len(fraud_indices),
        num_normal=sample_normal_size,
        aggr=aggr
    )
    print(f"Sampel embedding ({len(sample_labels)} sampel: {len(fraud_indices)} fraud + {sample_normal_size} normal) tersimpan ke: {emb_sample_path}")

    # 9. Phase 3: Klasifikasi dengan XGBoost
    print("\n--- [Phase 3: Pelatihan & Prediksi XGBoost Classifier] ---")
    num_neg_xgb = int((train_labels == 0).sum())
    num_pos_xgb = int((train_labels == 1).sum())
    scale_pos_weight = float(num_neg_xgb / num_pos_xgb) if num_pos_xgb > 0 else 1.0
    print(f"XGBoost scale_pos_weight: {scale_pos_weight:.2f}")
    
    cls_cfg = cfg.get('classifier', {})
    xgb_params = {
        'n_estimators': cls_cfg.get('n_estimators', 100),
        'max_depth': cls_cfg.get('max_depth', 5),
        'learning_rate': cls_cfg.get('learning_rate', 0.05),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    xgb_start = time.time()
    try:
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', device='cuda')
        xgb_model.fit(train_features, train_labels)
    except Exception as e:
        print(f"XGBoost GPU fit gagal ({e}), beralih ke CPU...")
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', n_jobs=-1)
        xgb_model.fit(train_features, train_labels)
    xgb_train_time = time.time() - xgb_start
    print(f"XGBoost training selesai dalam {xgb_train_time:.2f} detik.")

    # Simpan model XGBoost
    xgb_model_path = model_dir / f"exp22_{aggr}_xgb.json"
    xgb_model.save_model(xgb_model_path)
    print(f"Model XGBoost tersimpan di: {xgb_model_path}")

    # 10. Phase 4: Kalibrasi Threshold Dinamis pada Validation & Evaluasi Test
    print("\n--- [Phase 4: Kalibrasi Threshold Dinamis & Evaluasi Final] ---")
    # Prediksi probabilitas pada Validation
    val_probs = xgb_model.predict_proba(val_features)[:, 1]
    best_tau, val_tuning_stats = find_optimal_threshold(val_probs, val_labels)
    print(f"Optimal Threshold (tau*) dari Validation: {best_tau:.3f} (Val F1: {val_tuning_stats['val_f1']:.4f})")
    
    # Prediksi probabilitas pada Test
    test_pred_start = time.time()
    test_probs = xgb_model.predict_proba(test_features)[:, 1]
    inf_time_xgb = time.time() - test_pred_start
    total_test_inf_time = inf_time_test_gnn + inf_time_xgb

    # Evaluasi @ threshold 0.5 (Default komparasi baseline)
    preds_05 = (test_probs >= 0.5).astype(int)
    f1_05 = f1_score(test_labels, preds_05, zero_division=0)
    recall_05 = recall_score(test_labels, preds_05, zero_division=0)
    precision_05 = precision_score(test_labels, preds_05, zero_division=0)
    
    # Evaluasi @ threshold optimal tau* (Hasil utama)
    preds_tau = (test_probs >= best_tau).astype(int)
    f1_tau = f1_score(test_labels, preds_tau, zero_division=0)
    f1_macro_tau = f1_score(test_labels, preds_tau, average='macro', zero_division=0)
    recall_tau = recall_score(test_labels, preds_tau, zero_division=0)
    precision_tau = precision_score(test_labels, preds_tau, zero_division=0)
    accuracy_tau = accuracy_score(test_labels, preds_tau)
    
    # Metrik sensitivitas & spesifisitas
    cm = confusion_matrix(test_labels, preds_tau)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean = float(np.sqrt(recall_tau * specificity))
    else:
        gmean = 0.0

    # Metrik independen threshold (AUCPR & ROC-AUC)
    roc_auc = roc_auc_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0
    auprc = average_precision_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0

    # Metrik Operasional Perbankan: Lift@1% dan TP@500
    percentile = 0.01
    top_1pct_count = max(1, int(round(len(test_labels) * percentile)))
    top_1pct_idx = np.argsort(test_probs)[::-1][:top_1pct_count]
    precision_1pct = float(test_labels[top_1pct_idx].mean())
    base_fraud_rate = float(test_labels.mean())
    lift_1_percent = (precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

    top_500_count = min(500, len(test_labels))
    top_500_idx = np.argsort(test_probs)[::-1][:top_500_count]
    tp_at_500 = int(test_labels[top_500_idx].sum())
    precision_at_500 = float(tp_at_500 / top_500_count)

    total_pipeline_time = time.time() - total_pipeline_start

    # Tampilkan Ringkasan Hasil
    print("\n" + "="*80)
    print(f"                      HASIL EVALUASI EKSPERIMEN 22 ({aggr.upper()} AGGREGATOR)")
    print("="*80)
    print(f"Aggregator GNN      : {aggr.upper()} ({'Elementwise Maximization' if aggr=='max' else 'Mean Neighbor Averaging' if aggr=='mean' else 'Sum Neighbor Aggregation'})")
    print(f"Sampling Method     : {sampling_method.upper()}")
    print(f"Protokol Waktu      : SHINE FRAUD II (Train: {train_days}d, Val: {val_days}d, Gap: {gap_days}d, Test: {test_days}d)")
    print(f"Optimal Threshold   : tau* = {best_tau:.3f} (Validation F1 = {val_tuning_stats['val_f1']:.4f})")
    print("-"*80)
    print(f"Metrik @ Threshold 0.5 (Default Baseline):")
    print(f"  • F1-Score        : {f1_05:.4f}")
    print(f"  • Recall          : {recall_05:.4f} ({recall_05*100:.2f}%)")
    print(f"  • Precision       : {precision_05:.4f} ({precision_05*100:.2f}%)")
    print("-"*80)
    print(f"Metrik @ Optimal Threshold tau* = {best_tau:.3f} (Rekomendasi Utama):")
    print(f"  • F1-Score (Bin)  : {f1_tau:.4f}")
    print(f"  • F1-Score (Macro): {f1_macro_tau:.4f}")
    print(f"  • Recall          : {recall_tau:.4f} ({recall_tau*100:.2f}%)")
    print(f"  • Precision       : {precision_tau:.4f} ({precision_tau*100:.2f}%)")
    print(f"  • Accuracy        : {accuracy_tau:.4f}")
    print(f"  • GMean           : {gmean:.4f}")
    print("-"*80)
    print(f"Metrik Bebas Threshold & Metrik Operasional:")
    print(f"  • AUCPR (AUPRC)   : {auprc:.4f}")
    print(f"  • ROC-AUC         : {roc_auc:.4f}")
    print(f"  • Lift@1%         : {lift_1_percent:.4f}x")
    print(f"  • TP@500          : {tp_at_500}/{top_500_count} ({precision_at_500*100:.2f}%)")
    print("-"*80)
    print(f"Efisiensi Waktu Komputasi:")
    print(f"  • GNN Training    : {gnn_train_time:.2f} detik")
    print(f"  • XGBoost Train   : {xgb_train_time:.2f} detik")
    print(f"  • Test Inference  : {total_test_inf_time:.2f} detik (GNN: {inf_time_test_gnn:.2f}s, XGB: {inf_time_xgb:.2f}s)")
    print(f"  • Total Pipeline  : {total_pipeline_time:.2f} detik ({total_pipeline_time/60:.2f} menit)")
    print("="*80)

    # 11. Simpan Hasil ke Berkas JSON
    result_dict = {
        "experiment_id": "EXP22",
        "experiment_name": "GNN Aggregator Innovation: Elementwise Maximization vs Mean (SHINE)",
        "aggregator": aggr,
        "sampling_method": sampling_method,
        "time_window": {
            "reference_date": reference_date,
            "train_days": train_days,
            "val_days": val_days,
            "gap_days": gap_days,
            "test_days": test_days
        },
        "threshold_calibration": {
            "optimal_tau": float(best_tau),
            "val_f1": float(val_tuning_stats['val_f1']),
            "val_precision": float(val_tuning_stats['val_precision']),
            "val_recall": float(val_tuning_stats['val_recall'])
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_05),
            "recall": float(recall_05),
            "precision": float(precision_05)
        },
        "metrics_optimal_threshold": {
            "threshold": float(best_tau),
            "f1_score": float(f1_tau),
            "f1_macro": float(f1_macro_tau),
            "recall": float(recall_tau),
            "precision": float(precision_tau),
            "accuracy": float(accuracy_tau),
            "gmean": float(gmean)
        },
        "metrics_threshold_independent": {
            "auprc": float(auprc),
            "roc_auc": float(roc_auc),
            "lift_1_percent": float(lift_1_percent),
            "tp_at_500": int(tp_at_500),
            "precision_at_500": float(precision_at_500)
        },
        "runtime_seconds": {
            "data_preprocessing": float(data_prep_time),
            "gnn_training": float(gnn_train_time),
            "xgb_training": float(xgb_train_time),
            "gnn_test_inference": float(inf_time_test_gnn),
            "xgb_test_inference": float(inf_time_xgb),
            "total_test_inference": float(total_test_inf_time),
            "total_pipeline": float(total_pipeline_time)
        },
        "hyperparameters": {
            "hidden_channels": hidden_channels,
            "num_layers": num_layers,
            "aggr": aggr,
            "gnn_epochs": epochs,
            "gnn_batch_size": batch_size,
            "num_neighbors": num_neighbors,
            "xgb_n_estimators": xgb_params['n_estimators'],
            "xgb_max_depth": xgb_params['max_depth'],
            "xgb_learning_rate": xgb_params['learning_rate'],
            "undersampling_ratio": undersample_ratio
        }
    }
    
    result_path = result_dir / f"exp22_{aggr}_results.json"
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=4)
    print(f"\nHasil evaluasi lengkap tersimpan ke: {result_path}")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import json
import yaml
from pathlib import Path

def run_script(script_name, args=[]):
    print(f"\n{'='*75}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*75}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

def print_summary_table(result_dir: Path):
    aggregators = ['mean', 'max', 'sum']
    results = {}
    for aggr in aggregators:
        res_file = result_dir / f"exp22_{aggr}_results.json"
        if res_file.exists():
            with open(res_file, 'r') as f:
                results[aggr] = json.load(f)
                
    if not results:
        return
        
    print("\n" + "="*85)
    print("           RINGKASAN KOMPARASI HASIL EKSPERIMEN 22 (MEAN VS MAX)")
    print("="*85)
    print(f"{'Aggregator':<15} | {'Tau*':<6} | {'F1-Score':<8} | {'Recall':<8} | {'Precision':<9} | {'AUCPR':<8} | {'Lift@1%':<8} | {'TP@500':<7}")
    print("-" * 85)
    for aggr, res in results.items():
        tau = res['threshold_calibration']['optimal_tau']
        f1 = res['metrics_optimal_threshold']['f1_score']
        rec = res['metrics_optimal_threshold']['recall'] * 100
        prec = res['metrics_optimal_threshold']['precision'] * 100
        aucpr = res['metrics_threshold_independent']['auprc']
        lift = res['metrics_threshold_independent']['lift_1_percent']
        tp500 = res['metrics_threshold_independent']['tp_at_500']
        print(f"{aggr.upper():<15} | {tau:<6.3f} | {f1:<8.4f} | {rec:<7.2f}% | {prec:<8.2f}% | {aucpr:<8.4f} | {lift:<7.1f}x | {tp500:<7d}")
    print("="*85 + "\n")

if __name__ == "__main__":
    print("="*75)
    print("  MEMULAI EKSPERIMEN 22: GNN AGGREGATOR ABLATION (MEAN VS MAX)")
    print("="*75)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric, pyg-lib, matplotlib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse matplotlib -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric matplotlib > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install dependencies: {e}")
            os.system("pip install torch_geometric matplotlib > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        # Deteksi otomatis CSV di Kaggle
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
        result_dir_path = Path("/kaggle/working/result/exp22")
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
        result_dir_path = Path("/content/result/exp22")
    else:
        config_path = "config.reference.yaml"
        result_dir_path = Path("../../result/exp22")

    # 3. Jalankan eksperimen untuk agregator: Mean (Baseline) dan Max (Proposed)
    aggregators = ["mean", "max"]
    for aggr in aggregators:
        print(f"\n{'*'*75}")
        print(f"  >>> MULAI TRAINING & EVALUASI DENGAN AGGREGATOR: {aggr.upper()}")
        print(f"{'*'*75}\n")
        run_script("main.py", ["--config", config_path, "--aggr", aggr])
        
    # 4. Generate visualisasi perbandingan separabilitas t-SNE
    print(f"\n{'*'*75}")
    print("  >>> MEMBUAT VISUALISASI SEPARABILITAS t-SNE (MEAN VS MAX)")
    print(f"{'*'*75}\n")
    run_script("visualize_tsne.py", ["--result-dir", str(result_dir_path)])
    
    # 5. Tampilkan tabel ringkasan
    print_summary_table(result_dir_path)
    
    print("\n" + "="*75)
    print("  🎉 SELURUH PROSES PELATIHAN, EVALUASI & VISUALISASI EXP22 SELESAI! 🎉")
    print("="*75)



In [ ]:
!python run_all.py